# RSNA Knee Abnormality Detection — Phase 4C.6 corrected Phi GPU replay

Internet-off Kaggle notebook. **Do not enable internet. GPU is required.**
This notebook scores **synthetic reports only**. It does **not** read the 58
gold studies or the 4,349 report-only studies. It does **not** train an image
model, create folds, or submit.

Create a **new** kernel slug `rsna-knee-second-reader-phase46-replay`.
Do **not** overwrite gold kernel v7, canary v1, smoke v2, smoke/canary v1.3,
or constrained-canary kernel versions 1–3.

Prompt remains **v1.2.0**. Schema remains **v1.0.0**. Transport is **v1.4**.
Schema tokens are masked by a logits processor that records unconstrained
top-1 first. Do not pass `prefix_allowed_tokens_fn` into `model.generate`.
JSON-object stopper and Phi EOS IDs `[32000, 32007, 32001]` remain.

Gates: (1) runtime identity and processor-order proof; (2) paired replay of
`CANARY_SYNTHETIC_07`, `CANARY_SYNTHETIC_12`, `CANARY_SYNTHETIC_13`;
(3) 14-example unconstrained pass plus two constrained repeats.

If Gate 1 or Gate 2 fails, stop with `INTERVENTION_DIAGNOSTIC_FAIL`.
Never record `CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS`. If format, identity,
intervention accounting and repeatability pass, record
`CONSTRAINED_FORMAT_PASS_SEMANTIC_GATE_NOT_ESTABLISHED`.
`gold_rerun_authorized=false`. Do not execute notebook 07.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_second_reader_phase46_replay"
MARKERS = ("knee/config.py", "knee/second_reader_phase46.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks", "vendor", "third_party"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips: list[Path] = []
    dirs: list[Path] = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source() -> Path | None:
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips: list[Path] = []
    dirs: list[Path] = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        print(f"extracting_source_zip={zips[0]}")
        _safe_extract(zips[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        for nested in sorted(p for p in SRC_DST.iterdir() if p.is_dir()):
            if _looks_like_source(nested):
                return nested
    if dirs:
        print(f"copying_source_tree={dirs[0]}")
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
if SOURCE_ROOT is not None:
    resolved = str(SOURCE_ROOT.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"source_root={SOURCE_ROOT}")
else:
    print("source_root=not_found (will rely on an already importable knee package)")

try:
    from knee.kaggle_runtime import detect_kaggle_paths, KaggleLayout
    from knee.lm_format_enforcer_wheels import (
        install_from_local_wheels,
        runtime_dependency_versions,
        wheels_dir,
    )
    from knee.second_reader_constrained import ensure_lmfe_transformers_compat
    from knee.second_reader_phase46 import KAGGLE_PHASE46_OUTPUT_DIRNAME
except ImportError as exc:
    raise SystemExit(
        "PILOT STOPPED: could not import knee. Attach the private source dataset. "
        f"Original error: {exc}"
    ) from exc

print("internet_used=False gpu_required=True training_run=False process_report_only=False synthetic_canary=True constrained=True phase46=True")
print("gold_rerun_authorized=false gold_rerun_executed=false notebook_07_executed=false")
print("do_not_overwrite=mahdimzafari/rsna-knee-second-reader-constrained-canary v3")
print(detect_kaggle_paths(KaggleLayout(output_dirname=KAGGLE_PHASE46_OUTPUT_DIRNAME)))
wheel_root = wheels_dir(SOURCE_ROOT) if SOURCE_ROOT is not None else wheels_dir()
install_payload = install_from_local_wheels(wheel_root, root=SOURCE_ROOT)
print("lmfe_install=" + json.dumps({
    k: install_payload.get(k)
    for k in ("ok", "internet_used", "version", "license", "installed", "imported_versions")
}, sort_keys=True, default=str))
print("pip_check=" + json.dumps(install_payload.get("pip_check") or {}, sort_keys=True, default=str)[:4000])
print("runtime_versions=" + json.dumps(install_payload.get("runtime_versions") or runtime_dependency_versions(), sort_keys=True, default=str))
if not install_payload.get("ok"):
    raise SystemExit("WHEEL_INSTALL_FAIL")
compat = ensure_lmfe_transformers_compat()
print("lmfe_transformers_compat=" + json.dumps(compat, sort_keys=True))


In [ ]:
# Gate 1: hashes, processor order, schema audit. Do not print report text.
from pathlib import Path
import json
import time

from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
)
from knee.report_second_reader import inspect_generate_kwargs, locate_weights_directory
from knee.second_reader_constrained import (
    audit_schema_against_lmfe,
    schema_hash_provenance,
    tokenizer_prefix_tree_audit,
)
from knee.second_reader_json_stop import inspect_termination_from_snapshot
from knee.second_reader_phase46 import (
    FOCUS_EXAMPLE_IDS,
    PHASE46_KERNEL_SLUG,
    FakeSchemaMaskProcessor,
    gate1_file_hashes,
    git_head_sha,
    prove_processor_order,
    run_phase46_replay,
)
from knee.second_reader_pilot import load_pilot_config
from knee.second_reader_intervention_audit import semantic_gate_audit

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False
print("kernel_slug=" + PHASE46_KERNEL_SLUG)
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-constrained-canary")
print("gold_rerun_authorized=false")
print("process_report_only=false")
print("forbidden_outcome=CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS")
print("success_outcome_if_gates_pass=CONSTRAINED_FORMAT_PASS_SEMANTIC_GATE_NOT_ESTABLISHED")
print("git_head_sha=" + str(git_head_sha()))
print("focus_example_ids=" + json.dumps(list(FOCUS_EXAMPLE_IDS)))

cfg_v14 = None
if SOURCE_ROOT is not None:
    cfg_v14 = str(Path(SOURCE_ROOT) / "configs" / "second_reader_pilot_v1_4.yaml")
WEIGHTS = os.environ.get("SECOND_READER_WEIGHTS")
OUT_DIR = Path("/kaggle/working/rsna_second_reader_phase46_replay")
config = load_pilot_config(cfg_v14) if cfg_v14 else {}
weights_dir = locate_weights_directory(config.get("second_reader") or config, WEIGHTS)
print("weights_attached=" + str(weights_dir is not None))
if weights_dir is not None:
    inspection = inspect_termination_from_snapshot(weights_dir)
    print("termination_inspection=" + json.dumps(inspection, sort_keys=True, default=str)[:8000])

hashes = gate1_file_hashes(weights_dir=weights_dir)
print("gate1_file_hashes=" + json.dumps({k: hashes[k] for k in hashes if k != "wheel_manifest"}, sort_keys=True, default=str)[:8000])
if not hashes.get("ok"):
    raise SystemExit("GATE1_FAIL: file hash mismatch")
order = prove_processor_order()
print("processor_order=" + json.dumps(order, sort_keys=True))
if not order.get("ok"):
    raise SystemExit("GATE1_FAIL: processor-order proof")
un_kwargs = {"max_new_tokens": 2048, "do_sample": False, "eos_token_id": [32000, 32007, 32001]}
con_kwargs = dict(un_kwargs)
con_kwargs["logits_processor"] = [FakeSchemaMaskProcessor()]
print("inspect_unconstrained=" + json.dumps(inspect_generate_kwargs(un_kwargs), sort_keys=True))
print("inspect_constrained=" + json.dumps(inspect_generate_kwargs(con_kwargs), sort_keys=True))
print("semantic_gate=" + json.dumps(semantic_gate_audit(), sort_keys=True))
provenance = schema_hash_provenance()
print("schema_hash_provenance=" + json.dumps({
    "schema_raw_bytes_sha256": provenance["schema_raw_bytes_sha256"],
    "schema_canonical_json_sha256": provenance["schema_canonical_json_sha256"],
    "generation_schema_raw_bytes_sha256": provenance["generation_schema_raw_bytes_sha256"],
    "generation_schema_canonical_json_sha256": provenance["generation_schema_canonical_json_sha256"],
}, sort_keys=True))
schema_audit = audit_schema_against_lmfe()
print("schema_compatibility=" + json.dumps({k: schema_audit[k] for k in schema_audit if k != "fingerprint"}, sort_keys=True)[:4000])
if not schema_audit.get("ok"):
    raise SystemExit("SCHEMA_COMPATIBILITY_FAIL")
tok_dir = weights_dir
tokenizer_audit = tokenizer_prefix_tree_audit(tok_dir)
print("tokenizer_compatibility=" + json.dumps({k: tokenizer_audit[k] for k in tokenizer_audit if k not in {"skeleton"}}, sort_keys=True, default=str)[:4000])
if tokenizer_audit.get("skipped"):
    print("tokenizer_compatibility=skipped")
elif not tokenizer_audit.get("ok"):
    raise SystemExit("TOKENIZER_COMPATIBILITY_FAIL")
try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    print("gpu_name=" + gpu_name)
    if gpu_name == "cpu":
        raise SystemExit("GATE1_FAIL: cuda_required")
except Exception as exc:
    print("gpu_info=" + str(exc))


In [ ]:
# Gates 2 and 3: tokenize-once paired Phi replay. Completions are not printed.
started = time.time()
result = run_phase46_replay(
    output_dir=OUT_DIR,
    config_path=cfg_v14,
    weights_path=weights_dir,
)
elapsed = round(time.time() - started, 3)
peak_gpu = None
try:
    import torch
    if torch.cuda.is_available():
        peak_gpu = int(torch.cuda.max_memory_allocated())
except Exception:
    peak_gpu = None
public = {
    "status": result.get("status"),
    "ok": result.get("ok"),
    "skip_reason": result.get("skip_reason"),
    "gpu_replay_executed": result.get("gpu_replay_executed"),
    "semantic_gate_status": result.get("semantic_gate_status"),
    "gold_rerun_authorized": result.get("gold_rerun_authorized"),
    "gold_rerun_executed": result.get("gold_rerun_executed"),
    "n_gold_studies_processed": result.get("n_gold_studies_processed"),
    "n_report_only_studies_processed": result.get("n_report_only_studies_processed"),
    "kernel_slug": result.get("kernel_slug"),
    "inference_seconds": result.get("inference_seconds") or elapsed,
    "peak_gpu_memory_bytes": result.get("peak_gpu_memory_bytes") or peak_gpu,
    "gate1_ok": (result.get("gate1") or {}).get("ok"),
    "gate2_ok": (result.get("gate2") or {}).get("ok") if result.get("gate2") else None,
    "gate3": result.get("gate3"),
    "forbidden_outcome_not_used": result.get("forbidden_outcome_not_used"),
}
print("phase46=" + json.dumps(public, sort_keys=True, default=str)[:12000])
print("gold_rerun_authorized=false gold_rerun_executed=false notebook_07_executed=false")
if result.get("status") == "CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS":
    raise SystemExit("FORBIDDEN_OUTCOME: CONSTRAINED_FORMAT_PASS_SEMANTIC_PASS")
if not result.get("gpu_replay_executed"):
    raise SystemExit(str(result.get("skip_reason") or "gpu_replay_not_executed"))
if result.get("status") != "CONSTRAINED_FORMAT_PASS_SEMANTIC_GATE_NOT_ESTABLISHED":
    raise SystemExit(str(result.get("status") or "INTERVENTION_DIAGNOSTIC_FAIL"))
print("CONSTRAINED_FORMAT_PASS_SEMANTIC_GATE_NOT_ESTABLISHED")
